# Model Evaluation

This notebook plots different properties of the studied LLMs, focusing on their performance on translation tasks throughout their layers. Namely, in bold strokes, it looks at:

1. Models' translation accuracy on a parallel dataset (BLEU score)
2. Models' certainty at each layer, divided by part of speech (logitlens + entropy)
3. Models' accuracy of translation on the synthetic dataset

This notebook starts with the noun-adj dataset (English &rarr; French) and the model `meta-llama/Meta-Llama-3-8B`, comparing it with two translation-specific models: `facebook/mbart-large-50-many-to-many-mmt` and `google/madlad400-3b-mt`.

In [ ]:
!pip3 uninstall apex

In [ ]:
from transformers import LlamaForCausalLM, LlamaTokenizerFast, \
    MBartForConditionalGeneration, MBart50TokenizerFast, \
    T5ForConditionalGeneration, T5Tokenizer

from modeling.wrapper import LLamaWrapper

In [ ]:
CACHE_DIR = '/scratch/msonkin/word-order-thesis/cache/'

## Functions

In [ ]:
def load_model(model_name: str, model_class, tokenizer_class, cache_dir: str, device='cpu'):
    model = model_class.from_pretrained(model_name, cache_dir=cache_dir).to(device)
    tokenizer = tokenizer_class.from_pretrained(model_name, cache_dir=cache_dir)
    return model, tokenizer

In [ ]:
def get_translation_t5(sentence: str, lang_tag: str, model, tokenizer):
    text = f"{lang_tag} {sentence}"
    input_ids = tokenizer(text, return_tensors="pt").input_ids.to(model.device)
    outputs = model.generate(input_ids=input_ids)
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
def get_translation_mbart(sentence, src_tag, tgt_tag, model, tokenizer):
    tokenizer.src_lang = src_tag
    encoded_src = tokenizer(sentence, return_tensors="pt")
    generated_tokens = model.generate(
        **encoded_src,
        forced_bos_token_id=tokenizer.lang_code_to_id[tgt_tag]
    )
    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)

## Code

In [ ]:
t5_model, t5_tokenizer = load_model(
    'google/madlad400-3b-mt',
    T5ForConditionalGeneration,
    T5Tokenizer,
    cache_dir=CACHE_DIR,
    device='cuda'
    )

In [ ]:
mbart_model, mbart_tokenizer = load_model(
    'facebook/mbart-large-50-many-to-many-mmt',
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    cache_dir=CACHE_DIR,
    device='cuda:0'
    )

In [ ]:
print(get_translation_t5('Hello, how are you?', '<2de>', t5_model, t5_tokenizer))